# CELL 1: Install & Import Libraries 

In [1]:
!pip install mysql-connector-python sqlalchemy pymysql pandas
import pandas as pd
import numpy as np
import mysql.connector
from sqlalchemy import create_engine, text
import warnings
import time
 
warnings.filterwarnings('ignore')
print("All libraries imported successfully")
 

Defaulting to user installation because normal site-packages is not writeable
All libraries imported successfully


# CELL 2: Configuration

In [ ]:
DB_CONFIG = {
    'host'    : 'localhost',      
    'port'    : 3306,              
    'user'    : 'root',           
    'password': 'your_password_here',   
    'database': 'upi_fraud_db'
}

In [3]:
CSV_PATH   = '../data/upi_transactions_2024.csv'
CHUNK_SIZE = 10000  

In [4]:
print("Configuration set")
print(f"   Host     : {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"   Database : {DB_CONFIG['database']}")
print(f"   CSV      : {CSV_PATH}")
print(f"   Batch    : {CHUNK_SIZE:,} rows per insert")

Configuration set
   Host     : localhost:3306
   Database : upi_fraud_db
   CSV      : ../data/upi_transactions_2024.csv
   Batch    : 10,000 rows per insert


#  CELL 3: Load & Validate the CSV

In [5]:
print("Loading CSV into DataFrame...")
df = pd.read_csv(CSV_PATH)
 
print(f"\nDataset Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns        : {list(df.columns)}")


Loading CSV into DataFrame...

Dataset Shape  : 250,000 rows × 17 columns
Columns        : ['transaction id', 'timestamp', 'transaction type', 'merchant_category', 'amount (INR)', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']


###  Rename columns to match MySQL schema

In [6]:
df.rename(columns={
    'transaction id'  : 'transaction_id',
    'transaction type': 'transaction_type',
    'amount (INR)'    : 'amount_inr',
    'timestamp'       : 'txn_timestamp'
}, inplace=True)
 
print("\n Columns renamed:")
print(f"   {list(df.columns)}")
 


 Columns renamed:
   ['transaction_id', 'txn_timestamp', 'transaction_type', 'merchant_category', 'amount_inr', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']


### Validate data types

In [7]:
print("\n Data Types:")
print(df.dtypes)


 Data Types:
transaction_id        object
txn_timestamp         object
transaction_type      object
merchant_category     object
amount_inr             int64
transaction_status    object
sender_age_group      object
receiver_age_group    object
sender_state          object
sender_bank           object
receiver_bank         object
device_type           object
network_type          object
fraud_flag             int64
hour_of_day            int64
day_of_week           object
is_weekend             int64
dtype: object


### Convert timestamp column to datetime

In [8]:
df['txn_timestamp'] = pd.to_datetime(df['txn_timestamp'])
print(f"\n txn_timestamp converted → dtype: {df['txn_timestamp'].dtype}")
print(f"   Range: {df['txn_timestamp'].min()} → {df['txn_timestamp'].max()}")


 txn_timestamp converted → dtype: datetime64[ns]
   Range: 2024-01-01 00:05:10 → 2024-12-30 23:55:40


### Check for nulls (should be 0 for this clean dataset)

In [9]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f"\n Null Check — Total nulls: {total_nulls}")
if total_nulls == 0:
    print("No missing values — dataset is clean!")
else:
    print("Found nulls:")
    print(null_counts[null_counts > 0])


 Null Check — Total nulls: 0
No missing values — dataset is clean!


###  Validate target column

In [10]:
fraud_dist = df['fraud_flag'].value_counts()
print("\n Fraud_flag Distribution:")
print(f"   Legitimate (0) : {fraud_dist[0]:,}  ({fraud_dist[0]/len(df)*100:.2f}%)")
print(f"   Fraud      (1) : {fraud_dist[1]:,}  ({fraud_dist[1]/len(df)*100:.4f}%)")
print(f"\n  CLASS IMBALANCE ALERT: Only {fraud_dist[1]} fraud cases in {len(df):,} rows!")
print("   → When you reach the ML module, you MUST use SMOTE or class_weight='balanced'")
print("   → Never optimize Accuracy alone — use Precision, Recall, F1, AUC-PR")


 Fraud_flag Distribution:
   Legitimate (0) : 249,520  (99.81%)
   Fraud      (1) : 480  (0.1920%)

  CLASS IMBALANCE ALERT: Only 480 fraud cases in 250,000 rows!
   → When you reach the ML module, you MUST use SMOTE or class_weight='balanced'
   → Never optimize Accuracy alone — use Precision, Recall, F1, AUC-PR


# CELL 4: Connect to MySQL

In [11]:
print("🔌 Connecting to MySQL...")

 
try:
    # mysql.connector is for testing connection & schema verification
    conn = mysql.connector.connect(
        host     = DB_CONFIG['host'],
        port     = DB_CONFIG['port'],
        user     = DB_CONFIG['user'],
        password = DB_CONFIG['password'],
        database = DB_CONFIG['database']
    )
    cursor = conn.cursor()
 
    # Verify the table exists
    cursor.execute("SHOW TABLES LIKE 'upi_transactions';")
    result = cursor.fetchone()
    if result:
        print("Connected to MySQL | Table 'upi_transactions' found")
    else:
        print("Table 'upi_transactions' NOT FOUND")
        print("   → Run 01_schema_design.sql in MySQL Workbench first!")
 
    cursor.close()
    conn.close()
 
except mysql.connector.Error as err:
    print(f"MySQL connection failed: {err}")
    print("   → Check: Is MySQL server running?")
    print("   → Check: Is the password correct in DB_CONFIG?")
    print("   → Check: Does database 'upi_fraud_db' exist?")
 

🔌 Connecting to MySQL...
Connected to MySQL | Table 'upi_transactions' found


# CELL 5: Create SQLAlchemy Engine

In [12]:
engine = create_engine(
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}",
    echo=False   # set True to see generated SQL (useful for debugging)
)

### Test engine

In [13]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION()"))
    version = result.fetchone()[0]
    print(f"SQLAlchemy engine created | MySQL version: {version}")
 

SQLAlchemy engine created | MySQL version: 8.0.46


### Clear Existing Data from MySQL Table

In [14]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE upi_transactions"))

print("Table truncated successfully")

Table truncated successfully


### Verify the Table is Empty

In [15]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM upi_transactions",
    engine
)

,total_rows
0,0


# CELL 6: Load Data in Chunks

In [16]:
print(f"Starting data load → {len(df):,} rows in {CHUNK_SIZE:,}-row chunks...")
print(f"   Estimated chunks : {len(df) // CHUNK_SIZE + 1}")
print()
 
start_time  = time.time()
total_rows  = 0
chunk_count = 0
 
# Select only the columns that match MySQL table columns (in order)
columns_to_load = [
    'transaction_id', 'txn_timestamp', 'hour_of_day', 'day_of_week',
    'is_weekend', 'transaction_type', 'merchant_category', 'amount_inr',
    'transaction_status', 'sender_age_group', 'receiver_age_group',
    'sender_state', 'sender_bank', 'receiver_bank',
    'device_type', 'network_type', 'fraud_flag'
]
df_to_load = df[columns_to_load]
 
for chunk_start in range(0, len(df_to_load), CHUNK_SIZE):
 
    chunk = df_to_load.iloc[chunk_start : chunk_start + CHUNK_SIZE]
 
    chunk.to_sql(
        name      = 'upi_transactions',   # MySQL table name
        con       = engine,
        if_exists = 'append',             # append to existing schema
        index     = False,                # don't write DataFrame row index
        method    = 'multi',              # multi-row INSERT (faster than single)
        chunksize = CHUNK_SIZE
    )
 
    total_rows  += len(chunk)
    chunk_count += 1
    elapsed      = time.time() - start_time
 
    print(f"   Chunk {chunk_count:3d} | Rows loaded: {total_rows:>7,} / {len(df_to_load):,} "
          f"| Elapsed: {elapsed:.1f}s", end='\r')
 
elapsed_total = time.time() - start_time
print("\n\nData load complete!")
print(f"   Total rows inserted : {total_rows:,}")
print(f"   Total time          : {elapsed_total:.2f} seconds")
print(f"   Throughput          : {total_rows / elapsed_total:,.0f} rows/second")
 

Starting data load → 250,000 rows in 10,000-row chunks...
   Estimated chunks : 26

   Chunk  25 | Rows loaded: 250,000 / 250,000 | Elapsed: 191.0s

Data load complete!
   Total rows inserted : 250,000
   Total time          : 190.99 seconds
   Throughput          : 1,309 rows/second


# CELL 7: Verify Load — Row Count & Sample Data

In [17]:
print("Verifying data in MySQL...")
print("=" * 55)
 
with engine.connect() as conn:
 
    # Row count
    count = conn.execute(text("SELECT COUNT(*) FROM upi_transactions")).fetchone()[0]
    print(f"   Total rows in MySQL : {count:,}")
    print(f"   Rows in DataFrame   : {len(df):,}")
    print(f"   Match               : {'YES' if count == len(df) else '❌ NO — check for duplicates!'}")
 
    # Fraud distribution
    fraud_check = conn.execute(text("""
        SELECT
            fraud_flag,
            COUNT(*) AS count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 4) AS pct
        FROM upi_transactions
        GROUP BY fraud_flag
    """)).fetchall()
    print("\n   fraud_flag distribution in MySQL:")
    for row in fraud_check:
        label = 'Legitimate' if row[0] == 0 else 'FRAUD     '
        print(f"   {label} ({row[0]}) : {row[1]:,}  ({row[2]:.4f}%)")
 
    # Date range
    dates = conn.execute(text("""
        SELECT MIN(txn_timestamp), MAX(txn_timestamp)
        FROM upi_transactions
    """)).fetchone()
    print(f"\n   Date range : {dates[0]}  →  {dates[1]}")
 
    # Sample row
    sample = conn.execute(text("""
        SELECT transaction_id, txn_timestamp, transaction_type,
               amount_inr, sender_bank, receiver_bank, fraud_flag
        FROM upi_transactions
        LIMIT 3
    """)).fetchall()
    print("\n Sample rows:")
    for row in sample:
        print(f"   {row}")
 
print("\n DATA LOADING COMPLETE!")

Verifying data in MySQL...
   Total rows in MySQL : 250,000
   Rows in DataFrame   : 250,000
   Match               : YES

   fraud_flag distribution in MySQL:
   Legitimate (0) : 249,520  (99.8080%)
   FRAUD      (1) : 480  (0.1920%)

   Date range : 2024-01-01 00:05:10  →  2024-12-30 23:55:40

 Sample rows:
   ('TXN0000000001', datetime.datetime(2024, 10, 8, 15, 17, 28), 'P2P', 868, 'Axis', 'SBI', 0)
   ('TXN0000000002', datetime.datetime(2024, 4, 11, 6, 56), 'P2M', 1011, 'ICICI', 'Axis', 0)
   ('TXN0000000003', datetime.datetime(2024, 4, 2, 13, 27, 18), 'P2P', 477, 'Yes Bank', 'PNB', 0)

 DATA LOADING COMPLETE!


In [18]:
df.head()

,transaction_id,txn_timestamp,transaction_type,merchant_category,amount_inr,transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0
